In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.silver.sales")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.silver.sales (
    transaction_id STRING,
    date DATE,
    product_id STRING,
    customer_id STRING,
    store_id STRING,
    sold_qty INTEGER,
    unit_price DECIMAL(10,2)
)
""")

In [0]:
spark.sql("""
INSERT INTO de_mini_project.silver.sales
WITH sales_cleaned AS (
    SELECT 
        trxn_id_ AS transaction_id,
        COALESCE(
            try_to_date(_date_ref_, 'dd-MM-yyyy'),
            try_to_date(_date_ref_, 'MMM dd, yyyy'),
            try_to_date(_date_ref_, 'yyyy.MM.dd'),
            try_to_date(_date_ref_, 'dd-MMM-yy'),
            try_to_date(_date_ref_, 'yyyyMMdd'),
            try_to_date(_date_ref_, 'MM-dd-yyyy')
        ) AS date,
        prod_code_id AS product_id,
        CASE 
            WHEN TRIM(UPPER(cust_id_99)) = 'NULL' OR TRIM(cust_id_99) = '' OR cust_id_99 IS NULL 
            THEN 'UNKNOWN_CUSTOMER' 
            ELSE TRIM(cust_id_99) 
        END AS customer_id,
        store_loc_id AS store_id,
        CAST(qty_sold AS INTEGER) AS sold_qty,
        CASE 
            WHEN _unit_price_ LIKE '%$%' THEN 
                CAST(regexp_replace(_unit_price_, '[^0-9.]', '') AS DECIMAL(10,2)) * 93
            ELSE 
                CAST(regexp_replace(_unit_price_, '[^0-9.]', '') AS DECIMAL(10,2))
        END AS unit_price
    FROM de_mini_project.azure_blob_storage.sales
)
SELECT * FROM sales_cleaned
QUALIFY ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY date DESC) = 1
""")